# Notebook 2: Variational Monte Carlo from scratch

In this notebook, we will implement a complete variational Monte Carlo (VMC) algorithm from scratch using JAX and apply it to the one-dimensional transverse-field Ising (TFI) model.

The goal is not to build the most efficient implementation possible, but rather to understand the fundamental ingredients underlying modern neural-quantum-state algorithms. Starting from a simple variational wavefunction, we will progressively construct all the components required for VMC. By the end of the notebook, we will have assembled a minimal but complete VMC engine and will be able to compare ordinary gradient descent with stochastic reconfiguration on a concrete quantum many-body problem.

## 1. Transverse-field Ising model

Throughout this notebook, we consider the following one-dimensional transverse-field Ising model:

$\hat{H} = -\sum_{i=1}^L \hat{\sigma}_i^z \hat{\sigma}_{i+1}^z - g \sum_{i=1}^L \hat{\sigma}_i^x .$

An important point is that, with the above sign convention, the off-diagonal matrix elements generated by the transverse field are all non-positive. By the Perron-Frobenius theorem, the ground-state wavefunction can thus be chosen to be real and non-negative in the computational basis. To simplify matters, we then focus exclusively on modelling the amplitudes of the wavefunction and do not need to consider complex phases.

For pedagogical purposes, we will work with relatively small system sizes. This will allow us to compare Monte Carlo estimates against exact sums over all ($2^L$) spin configurations, providing useful sanity checks as we build the VMC algorithm.

In [100]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import jax
import jax.numpy as jnp

L = 12
g = 1.0

A spin configuration is represented by a vector

$$\sigma = (\sigma_1,\ldots,\sigma_L), \qquad \sigma_i\in\{-1,+1\}.$$

For small systems, we can explicitly enumerate all ($2^L$) configurations.

In [60]:
def all_configurations(L, dtype=jnp.int8):
    """Enumerate all spin configurations σ_i ∈ {-1, +1}."""
    states = jnp.arange(2**L)[:, None]
    bits = (states >> jnp.arange(L)) & 1
    return (2 * bits - 1).astype(dtype)

configs = all_configurations(L)

# configs.shape == (2**L, L)
print(configs.shape)

# First five configurations
configs[:5]

(4096, 12)


Array([[-1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
       [ 1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
       [-1,  1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
       [ 1,  1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
       [-1, -1,  1, -1, -1, -1, -1, -1, -1, -1, -1, -1]], dtype=int8)

## 2. Trial wavefunction

In VMC, we represent the quantum state through a parametrised wavefunction $\psi_{\theta}(\sigma)$. Since the ground state of the transverse-field Ising model can be chosen positive in the computational basis, we will directly parametrise the logarithm of the wavefunction amplitude, $\log \psi_{\theta}(\sigma)$. Working with $\log\psi_{\theta}$ is numerically convenient and will also help us write numerically stable code.

We use a restricted Boltzmann machine ansatz:

$$\log \psi_{\theta}(\sigma) = \sum_i a_i \sigma_i + \sum_j \log\cosh\left(b_j + \sum_i W_{ij}\sigma_i\right),$$

where the parameters are the visible biases $a_i$, hidden biases $b_j$, and weights $W_{ij}$. 

In JAX, model parameters are typically organised into *Pytrees*: nested Python containers (such as dictionaries, lists, or tuples) whose leaves are JAX arrays. Pytrees provide a convenient abstraction for working with large collections of parameters, as JAX transformations such as automatic differentiation, vectorisation, and optimisation naturally operate on them.

For the RBM, we store the visible biases `a`, hidden biases `b`, and weight matrix `W` in a dictionary. Below, we can generate a set of randomly initialised parameters in a way very similar to that of the literature:

In [61]:
def init_rbm_params(key, L, alpha=2, stddev=0.01):
    M = alpha * L

    k1, k2, k3 = jax.random.split(key, 3)

    return {
        "a": stddev * jax.random.normal(k1, (L,)),
        "b": stddev * jax.random.normal(k2, (M,)),
        "W": stddev * jax.random.normal(k3, (L, M)),
    }

key = jax.random.PRNGKey(0)
key, subkey = jax.random.split(key)

params = init_rbm_params(subkey, L)

One can apply functions to all leaves of a Pytree with `jax.tree.map`. Let us use that to show the structure and shape of the set of parameters:

In [62]:
print("Parameters shape: ", jax.tree.map(lambda x: x.shape, params))

n_params = sum(x.size for x in jax.tree.leaves(params))
print(f"Number of parameters: {n_params}")

print("Hilbert-space dimension: ", configs.shape[0]) 

Parameters shape:  {'W': (12, 24), 'a': (12,), 'b': (24,)}
Number of parameters: 324
Hilbert-space dimension:  4096


Let us now code the RBM wavefunction that takes a Pytree of parameters and a single configuration and produces the corresponding wavefunction amplitude:

In [63]:
def logpsi_single(params, sigma):
    # shapes: (alpha * L,) + (L,) @ (L, alpha * L) -> (alpha * L,)
    theta = params["b"] + sigma @ params["W"]
    # shapes: (L,) @ (L,) + scalar -> scalar
    logvalue = sigma @ params["a"] + jnp.sum(jnp.log(jnp.cosh(theta)))
    return logvalue

config = configs[0]
print("config: ", config)
print("logpsi: ", logpsi_single(params, configs[0]))

config:  [-1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1]
logpsi:  0.024511285


We now want to evaluate the wavefunction on all configurations in our batch (here, all configurations of the Hilbert space).

In the present case, a manually vectorised implementation would be straightforward because matrix multiplication naturally supports batching. However, deriving batched implementations quickly becomes tedious for more complicated functions.

Instead, JAX provides the `vmap` transformation, which automatically lifts a function acting on a single configuration into a function acting on a batch of configurations.

In [64]:
logpsi = jax.vmap(logpsi_single, in_axes=(None, 0))
logvalues = logpsi(params, configs)

print(logvalues.shape)
logvalues

(4096,)


Array([0.02451129, 0.0335076 , 0.01050396, ..., 0.03356557, 0.01938985,
       0.01737342], dtype=float32)

The argument `in_axes=(None, 0)` means do not vectorise over `params` but do vectorise over the leading axis of `configs`.

From now on, we will use this vectorised version whenever we need to evaluate the wavefunction on many configurations simultaneously.

## 3. Exact *full-sum* expectation values

Given a wavefunction $\psi_\theta(\sigma)$, we can construct the Born probability distribution

$$p_\theta(\sigma) = \frac{|\psi_\theta(\sigma)|^2}{\sum_{\sigma'} |\psi_\theta(\sigma')|^2}.$$

Since we have enumerated all configurations, we can compute this distribution exactly.

In [65]:
logvalues = logpsi(params, configs)

def logprobs(logvalues, normalize=True):
    _logprobs = 2 * jnp.real(logvalues)
    if normalize:
        # Stable normalisation
        _logprobs -= jax.scipy.special.logsumexp(_logprobs)
    return _logprobs

probs = jnp.exp(logprobs(logvalues))

print("normalisation:", probs.sum())

normalisation: 0.99999994


We can readily evaluate some physical quantities on our first variational state! For instance, let us consider the total magnetisation $M_z(\sigma) = \frac{1}{L}\sum_i \sigma_i$ of the state. The expectation value can now be computed exactly as

$$\langle \hat{M}_z \rangle = \sum_\sigma p_\theta(\sigma)\, M_z(\sigma).$$

In [66]:
magnetisation = jnp.mean(configs, axis=-1)

m = jnp.sum(probs * magnetisation)

print(m)

-0.0005851388


## 4. Local energy

Unlike the magnetisation, the Hamiltonian is not diagonal in the computational basis because of the transverse-field term. The variational energy therefore cannot be written directly as the classical expectation value of a function of the spin configuration. A central idea of variational Monte Carlo is that the energy can nevertheless be expressed as an expectation value over the Born probability distribution through the introduction of the local energy:

$$E_{\mathrm{loc}}(\sigma) = \sum_{\sigma'} H_{\sigma,\sigma'} \frac{\psi_\theta(\sigma')}{\psi_\theta(\sigma)}.$$

Then, one has

$$E_\theta = \sum_\sigma p_\theta(\sigma) E_{\mathrm{loc}}.$$

The local energy is therefore the central quantity that must be evaluated during a VMC simulation.

Importantly, despite the apparent summation over all configurations $\sigma'$ in the above definition, for any given configuration $\sigma$, one can restrict that sum only to its connected elements $\sigma'$ through the Hamiltonian $\hat{H}$, that is such that $H_{\sigma,\sigma'} := \langle \sigma|\hat{H}|\sigma'\rangle \neq 0$.

To evaluate the local energy on a configuration $\sigma$, we need to know which configurations $\sigma'$ are connected by the Hamiltonian and their contribution $H_{\sigma,\sigma'}$ to $E_\mathrm{loc}(\sigma)$. For the transverse-field Ising model, there are two types of contributions:

1. The diagonal interaction term connects $\sigma$ to itself,

$$\sigma'=\sigma, \qquad H_{\sigma,\sigma} = -\sum_i \sigma_i\sigma_{i+1}.$$

2. The transverse-field term flips one spin at a time. If $\sigma^{(i)}$ denotes the configuration obtained from $\sigma$ by flipping spin $i$, then we have the following pairs of connected elements and their corresponding matrix elements:
$$\lbrace (\sigma^{(i)}, -g)\rbrace_{i=1}^{L}$$
Thus, each configuration is connected to itself and to the $L$ configurations obtained by single-spin flips, with the same local-energy contribution $-g$.

Let us code this:

In [67]:
def get_mels_and_conns(sigma, g):
    sigma_shifted = jnp.roll(sigma, -1)
    e_diag = -jnp.sum(sigma * sigma_shifted)

    # matrix elements
    matrix_elements = jnp.concatenate([
        jnp.array([e_diag]),  # diagonal
        -g * jnp.ones(L),     # off-diagonal
    ])

    # -1 on the diagonal, +1 everywhere else
    flip_masks = 1 - 2 * jnp.eye(L, dtype=sigma.dtype)
    
    # connected elements
    connected = jnp.concatenate([
        sigma[None, :],       # identity
        sigma * flip_masks,   # one-spin flipped
    ])

    return matrix_elements, connected

# Let us try:
get_mels_and_conns(configs[0], g)

(Array([-12.,  -1.,  -1.,  -1.,  -1.,  -1.,  -1.,  -1.,  -1.,  -1.,  -1.,
         -1.,  -1.], dtype=float32),
 Array([[-1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
        [ 1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
        [-1,  1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
        [-1, -1,  1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
        [-1, -1, -1,  1, -1, -1, -1, -1, -1, -1, -1, -1],
        [-1, -1, -1, -1,  1, -1, -1, -1, -1, -1, -1, -1],
        [-1, -1, -1, -1, -1,  1, -1, -1, -1, -1, -1, -1],
        [-1, -1, -1, -1, -1, -1,  1, -1, -1, -1, -1, -1],
        [-1, -1, -1, -1, -1, -1, -1,  1, -1, -1, -1, -1],
        [-1, -1, -1, -1, -1, -1, -1, -1,  1, -1, -1, -1],
        [-1, -1, -1, -1, -1, -1, -1, -1, -1,  1, -1, -1],
        [-1, -1, -1, -1, -1, -1, -1, -1, -1, -1,  1, -1],
        [-1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,  1]], dtype=int8))

Now that we have the matrix and connected elements, we can code the local energy in terms of log amplitudes:

In [68]:
def local_energy_single(params, sigma, g):
    matrix_elements, connected = get_mels_and_conns(sigma, g)

    # evaluate the logvalues entering the ratio
    logpsi_sigma = logpsi_single(params, sigma)  # /!\ note that logpsi is defined globally
    logpsi_connected = logpsi(params, connected) # to reduce clutter.

    # numerically stable ratio
    ratios = jnp.exp(logpsi_connected - logpsi_sigma)

    return jnp.sum(matrix_elements * ratios)

config = configs[0]
print(local_energy_single(params, config, g))

-23.969437


Then, we vectorise:

In [69]:
local_energy = jax.vmap(local_energy_single, in_axes=(None, 0, None))

local_energies = local_energy(params, configs, g)
local_energies, local_energies.shape

(Array([-23.969439, -19.940563, -19.99521 , ..., -19.949451, -19.968884,
        -23.98394 ], dtype=float32),
 (4096,))

## 5. Exact variational energy

We now have all ingredients needed to compute the variational energy exactly for our small system:

$$E_\theta = \sum_\sigma p_\theta(\sigma)\,E_{\mathrm{loc}}(\sigma).$$

We can minimally code this as follows:

In [70]:
def energy_fullsum(params, configs, g):
    logvalues = logpsi(params, configs)  # defined globally
    probs = jnp.exp(logprobs(logvalues, normalize=True))

    # here, technically, we could avoid recomputing the logvalues
    local_energies = local_energy(params, configs, g)

    return jnp.sum(probs * local_energies)

energy = energy_fullsum(params, configs, g)
print("Energy:", energy)
print("Energy per site:", energy / L)

Energy: -12.003435
Energy per site: -1.0002862


This exact full sum will serve as a reference when we replace the sum over all configurations by Monte Carlo sampling in the next section. Before jumping to Metropolis-Hastings, let us first see how the full summation above can be replaced by a Monte Carlo integration of the form:

$$\hat{E}_\theta = \frac{1}{N_s}\sum_{n=1}^{N_s} E_\mathrm{loc}(\sigma^{(i)}),
\qquad
\sigma^{(i)} \sim p_\theta$$

For this small system, we know the full probability distribution $p_\theta(\sigma)$. We can therefore draw independent samples from it directly using `jax.random.choice`:

In [71]:
def sample_direct(params, configs, key, n_samples):
    N = configs.shape[0]  # Hilbert-space dimension
    logvalues = logpsi(params, configs)   # defined globally
    probs = jnp.exp(logprobs(logvalues))
    sample_ids = jax.random.choice(
        key, 
        N,
        shape=(n_samples,),
        p=probs,
    )
    samples = configs[sample_ids]
    return samples

n_samples = 2 * 1024
key, subkey = jax.random.split(key, 2)

samples = sample_direct(params, configs, subkey, n_samples)
print(samples.shape)
print(samples[:5])

(2048, 12)
[[ 1  1 -1 -1 -1  1  1 -1 -1  1  1  1]
 [ 1 -1  1 -1  1 -1 -1  1 -1  1 -1  1]
 [ 1 -1 -1  1  1  1  1  1  1 -1  1  1]
 [ 1 -1  1 -1  1 -1 -1 -1  1  1 -1 -1]
 [-1 -1 -1 -1  1  1  1  1 -1 -1  1  1]]


In [81]:
def energy_mc(params, local_energy, g, samples):
    local_energies = local_energy(params, samples, g)

    n_samples = samples.shape[0]
    e_mean = jnp.mean(local_energies)
    e_err = jnp.std(local_energies) / jnp.sqrt(n_samples)
    
    return e_mean, e_err

energy_sampled, energy_error = energy_mc(params, local_energy, g, samples)
print("Sampled energy:      ", energy_sampled)
print("Statistical error:   ", energy_error)
print("Full-sum energy:     ", energy)

Sampled energy:       -12.0873575
Statistical error:    0.03749452
Full-sum energy:      -12.003435


The sampled estimate fluctuates around the exact full-sum value, with an error that decreases as $1/\sqrt{N_s}$. In realistic VMC calculations, we do not have access to independent samples from the exact Born distribution. The role of Metropolis-Hastings will be to generate approximate samples from the same distribution without enumerating the Hilbert space.

## 6. Metropolis-Hastings sampling

In realistic VMC calculations, we cannot enumerate all configurations and we cannot sample directly from the exact Born distribution. Instead, we construct a Markov chain whose stationary distribution is

$$p_\theta(\sigma) \propto |\psi_\theta(\sigma)|^2.$$

We will use the Metropolis-Hastings algorithm with simple single-spin-flip proposals. Starting from a configuration $\sigma$, we propose a new configuration $\sigma'$ by flipping one randomly chosen spin. The move is accepted with probability

$$A(\sigma \to \sigma') = \min\left(1, \frac{p_\theta(\sigma')}{p_\theta(\sigma)}\right) = \min\left(1, \left|\frac{\psi_\theta(\sigma')}{\psi_\theta(\sigma)}\right|^2\right).$$

This can be implemented in a numerically stable manner by computing log ratios

$$\log \frac{p_\theta(\sigma')}{p_\theta(\sigma)} = 2\mathrm{Re}\left[\log\psi_\theta(\sigma')-\log\psi_\theta(\sigma)\right].$$

Let's see:

In [76]:
def metropolis_step_python(key, params, sigma):
    key_site, key_accept = jax.random.split(key)

    # pick a random site
    i = jax.random.randint(key_site, shape=(), minval=0, maxval=L)
    # flip the spin
    sigma_proposed = sigma.at[i].multiply(-1)

    # log acceptance rate: log A
    logp_ratio = 2 * (
        logpsi_single(params, sigma_proposed)
        - logpsi_single(params, sigma)
    )

    # accept with probability A
    accept = jnp.log(jax.random.uniform(key_accept)) < logp_ratio

    # update the configuration if accepted
    sigma_next = jnp.where(accept, sigma_proposed, sigma)

    return sigma_next, accept

key, subkey = jax.random.split(key)

config = configs[0]
config, accept = metropolis_step_python(subkey, params, config)

print(config)
print(accept)

[-1 -1 -1 -1 -1  1 -1 -1 -1 -1 -1 -1]
True


We can now build the Markov chain:

In [82]:
def metropolis_chain_python(key, params, sigma, n_steps):
    samples = []
    accepts = []

    for _ in range(n_steps):
        key, subkey = jax.random.split(key)
        sigma, accept = metropolis_step_python(subkey, params, sigma)

        samples.append(sigma)
        accepts.append(accept)

    return jnp.array(samples), jnp.array(accepts)

n_steps = 8 * 1024

key, subkey = jax.random.split(key)

# Starting point of the chain
sigma0 = configs[0]
samples, accepts = metropolis_chain_python(subkey, params, sigma0, n_steps)

print(samples.shape)
# The randomly initialised wavefunction is quite flat so this should be quite high
print("Acceptance rate:", jnp.mean(accepts))

(8192, 12)
Acceptance rate: 0.9863281


In [84]:
energy_mcmc, energy_error = energy_mc(params, local_energy, g, samples)

print("MCMC energy:       ", energy_mcmc)
print("Statistical error:", energy_error)
print("Exact energy:     ", energy)

MCMC energy:        -12.161955
Statistical error: 0.037555028
Exact energy:      -12.003435


This implementation is intentionally simple and mirrors the textbook Metropolis-Hastings algorithm. However, it uses a Python loop and repeatedly appends to lists, so it is not suitable for efficient JAX execution. Indeed, if decorated with `jax.jit`, the for loop would be unrolled before compiling the function, thereby leading to a compile time that would linearly grow with the number of samples.

Next, we rewrite the same Markov-chain update using `jax.lax.scan`, which allows JAX to compile the entire sampling loop. Additionally, we vectorise it across an arbitrary number of independent Markov chains that will be updated simultaneously. Besides performance reasons, this reduces a bit the potential bias of Monte Carlo estimation. Indeed, since chains are independent, interchain samples are less correlated than intrachain ones.

You don't necessarily have to understand everything in this code!

In [86]:
def random_state(key, n_chains, L, dtype=jnp.int8):
    """Generate random spin configurations σᵢ ∈ {-1, +1}."""
    bits = jax.random.bernoulli(key, p=0.5, shape=(n_chains, L))
    return (2 * bits - 1).astype(dtype)

def metropolis_chains(key, params, sigma, n_samples_per_chain):
    """
    Runs several Metropolis-Hastings chains in parallel.

    sigma has shape (n_chains, L).
    Returns samples with shape (n_samples_per_chain, n_chains, L).
    """

    def step(carry, _):
        key, sigma = carry

        key, key_site, key_accept = jax.random.split(key, 3)

        n_chains = sigma.shape[0]

        sites = jax.random.randint(
            key_site,
            shape=(n_chains,),
            minval=0,
            maxval=L,
        )

        rows = jnp.arange(n_chains)
        sigma_proposed = sigma.at[rows, sites].multiply(-1)

        logp_ratio = 2 * (
            logpsi(params, sigma_proposed)
            - logpsi(params, sigma)
        )

        accept = jnp.log(jax.random.uniform(key_accept, shape=(n_chains,))) < logp_ratio

        sigma = jnp.where(
            accept[:, None],
            sigma_proposed,
            sigma,
        )

        return (key, sigma), (sigma, accept)

    (_, sigma), (samples, accepts) = jax.lax.scan(
        step,
        init=(key, sigma),
        xs=None,
        length=n_steps,
    )

    return samples, accepts, sigma

metropolis_chains = jax.jit(
    metropolis_chains,
    static_argnames=("n_samples_per_chain",),
)

In [87]:
n_chains = 16
n_samples_per_chain = 1024

key, subkey = jax.random.split(key)
sigma0 = random_state(subkey, n_chains, L)

key, subkey = jax.random.split(key)
samples, accepts, sigma = metropolis_chains(
    subkey,
    params,
    sigma0,
    n_samples_per_chain,
)

print("samples shape:", samples.shape)
print("accepts shape:", accepts.shape)
print("final state shape:", sigma.shape)
print("acceptance rate:", jnp.mean(accepts))

samples shape: (8192, 16, 12)
accepts shape: (8192, 16)
final state shape: (16, 12)
acceptance rate: 0.9859848


In [90]:
energy_mcmc, energy_error = energy_mc(params, local_energy, g, samples.reshape(-1, L))

print("MCMC energy:       ", energy_mcmc)
print("Statistical error:", energy_error)
print("Exact energy:     ", energy)

MCMC energy:        -12.005976
Statistical error: 0.0096016005
Exact energy:      -12.003435


## 7. Variational-energy gradients

To optimise the variational parameters, we need the gradient of the energy with respect to the parameters. A direct differentiation of the variational energy leads to the following estimator:

$$\frac{\partial E}{\partial \theta_k} = 2\left(\langle O_k E_{\mathrm{loc}} \rangle_p - \langle O_k \rangle_p \langle E_{\mathrm{loc}} \rangle_p \right),$$

where

$$O_k(\sigma) = \frac{\partial \log\psi_\theta(\sigma)}{\partial \theta_k}$$

is known as the logarithmic derivative of the wavefunction. The gradient therefore takes the form of a covariance between the local energy and the logarithmic derivatives.

The logarithmic derivatives can be computed automatically using JAX's automatic differentiation.

In [92]:
logpsi_grad = jax.grad(logpsi_single)

sigma = configs[0]
# log derivatives wrt params
grads = logpsi_grad(params, sigma)

# Same shape as the parameters!
print(jax.tree.map(lambda x: x.shape, grads))

{'W': (12, 24), 'a': (12,), 'b': (24,)}


We can of course vectorise this over the batch (leading) dimension:

In [95]:
logpsi_grad_batch = jax.vmap(
    logpsi_grad,
    in_axes=(None, 0),
)

# fuse (n_samples_per_chain, n_chains, L) into (n_samples, L)
samples_flat = samples.reshape(-1, L)

O = logpsi_grad_batch(params, samples_flat)

print(jax.tree.map(lambda x: x.shape, O))

{'W': (131072, 12, 24), 'a': (131072, 12), 'b': (131072, 24)}


In [96]:
def estimate_energy_and_grad(params, samples):
    eloc = local_energy(params, samples)
    energy = jnp.mean(eloc)

    def energy_from_samples(params):
        eloc = local_energy(params, samples)
        return jnp.mean(eloc)

    grad = jax.grad(energy_from_samples)(params)

    return energy, grad

In [98]:
def apply_update(params, grad, learning_rate):
    return jax.tree.map(
        lambda p, g: p - learning_rate * g,
        params,
        grad,
    )

In [99]:
n_iter = 100
learning_rate = 0.05

energies = []

for it in range(n_iter):
    key, subkey = jax.random.split(key)

    samples, accepts, sigma = metropolis_chains(
        subkey,
        params,
        sigma,
        n_steps,
    )

    samples_flat = samples.reshape(-1, L)

    energy, grad = estimate_energy_and_grad(params, samples_flat)

    params = apply_update(params, grad, learning_rate)

    energies.append(energy)

    if it % 10 == 0:
        print(f"it={it:03d} E/L={energy / L:.6f} acc={jnp.mean(accepts):.3f}")

IndexError: Too many indices: array is 1-dimensional, but 2 were indexed